# Exercício 1 — Ler o valor como número

In [27]:
import pandas as pd

df = pd.read_csv("../data/receitas_candidatos_2026_RS.csv", sep=";", decimal=",", encoding="latin-1")

df["VR_RECEITA"]

0        25000.0
1        25000.0
2         2250.0
3         2000.0
4         9000.0
          ...   
14036    10000.0
14037     5000.0
14038     5000.0
14039    50000.0
14040     2500.0
Name: VR_RECEITA, Length: 14041, dtype: float64

# Exercício 2 — Conhecer a base

In [28]:
print(f"O dataframe tem {df.shape[0]} linhas e {df.shape[1]} colunas.")

O dataframe tem 14041 linhas e 60 colunas.


In [29]:
df.info() 

<class 'pandas.DataFrame'>
RangeIndex: 14041 entries, 0 to 14040
Data columns (total 60 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   DT_GERACAO                     14041 non-null  str    
 1   HH_GERACAO                     14041 non-null  str    
 2   AA_ELEICAO                     14041 non-null  int64  
 3   CD_TIPO_ELEICAO                14041 non-null  int64  
 4   NM_TIPO_ELEICAO                14041 non-null  str    
 5   CD_ELEICAO                     14041 non-null  int64  
 6   DS_ELEICAO                     14041 non-null  str    
 7   DT_ELEICAO                     14041 non-null  str    
 8   ST_TURNO                       14041 non-null  int64  
 9   TP_PRESTACAO_CONTAS            14041 non-null  str    
 10  DT_PRESTACAO_CONTAS            14041 non-null  str    
 11  SQ_PRESTADOR_CONTAS            14041 non-null  int64  
 12  SG_UF                          14041 non-null  str    
 1

In [30]:
nulos = df.isnull().sum()
nulos[nulos > 0]

DT_RECEITA    79
dtype: int64

In [31]:
serie_candidatos = df["NM_CANDIDATO"].nunique()
print(f"Existem {serie_candidatos} candidatos diferentes na base")

Existem 995 candidatos diferentes na base


In [32]:
serie_partidos = df["SG_PARTIDO"].nunique()
print(f"Existem {serie_partidos} partidos diferentes na base")

Existem 27 partidos diferentes na base


# Exercício 3 — Ficar só com as colunas que interessam

In [33]:
colunas = ["NM_CANDIDATO", 
           "SG_PARTIDO", 
           "DS_CARGO", 
           "DS_GENERO", 
           "DS_FONTE_RECEITA", 
           "DS_ORIGEM_RECEITA", 
           "DT_RECEITA", 
           "VR_RECEITA"
           ]

df_receitas = df[colunas]

# Exercício 4 — De onde vem o dinheiro

In [34]:
df_receitas["DS_FONTE_RECEITA"].value_counts()

DS_FONTE_RECEITA
OUTROS RECURSOS     9489
FUNDO ESPECIAL      3820
FUNDO PARTIDARIO     653
#NULO                 79
Name: count, dtype: int64

# Exercício 5 — Isolar o fundão

In [35]:
filtro_fundao_eleitoral = df_receitas["DS_FONTE_RECEITA"] == "FUNDO ESPECIAL"
df_receitas[filtro_fundao_eleitoral]

print(f"Temos {df_receitas[filtro_fundao_eleitoral].shape[0]} receitas do fundão eleitoral")

Temos 3820 receitas do fundão eleitoral


# Exercício 6 — Quanto é o fundão

In [36]:
df_fundao = df_receitas[filtro_fundao_eleitoral]
df_fundao["VR_RECEITA"].sum().round(2)

np.float64(256273993.78)

In [37]:
vr_receita_total = df_receitas["VR_RECEITA"].sum()
vr_receita_fundao = df_fundao["VR_RECEITA"].sum()
percentual_fundao = (vr_receita_fundao / vr_receita_total) *100

print(f"O valor de fundão eleitoral foi {percentual_fundao:.1f}% do total de receitas")

O valor de fundão eleitoral foi 79.9% do total de receitas


# Exercício 7 — Fundão por cargo

In [38]:
lista_cargo = df_fundao["DS_CARGO"].unique()

for cargo in lista_cargo:
    filtro_cargo = df_fundao["DS_CARGO"] == cargo
    df_filtro_cargo = df_fundao[filtro_cargo]
    valor_total_cargo = df_filtro_cargo["VR_RECEITA"].sum()
    print(f"{cargo}: R$ {valor_total_cargo:,.2f}")

Deputado Estadual: R$ 80,023,644.98
Deputado Federal: R$ 128,010,529.34
Senador: R$ 19,147,994.81
Governador: R$ 29,091,824.65


# Exercício 8 — As cinco maiores transferências do fundão

In [39]:
df_fundao.sort_values("VR_RECEITA", ascending= False).head(5)

,NM_CANDIDATO,SG_PARTIDO,DS_CARGO,DS_GENERO,DS_FONTE_RECEITA,DS_ORIGEM_RECEITA,DT_RECEITA,VR_RECEITA
5846,JULIANA BRIZOLA,PDT,Governador,Feminino,FUNDO ESPECIAL,Recursos de partido político,24/08/2026,10000000.0
10559,LUCIANO LORENZINI ZUCCO,PL,Governador,Masculino,FUNDO ESPECIAL,Recursos de partido político,24/08/2026,5000000.0
2310,UBIRATAN ANTUNES SANDERSON,PL,Senador,Masculino,FUNDO ESPECIAL,Recursos de partido político,24/08/2026,4400000.0
3984,GABRIEL VIEIRA DE SOUZA,MDB,Governador,Masculino,FUNDO ESPECIAL,Recursos de partido político,04/09/2026,4000000.0
3007,GERMANO ANTONIO RIGOTTO,MDB,Senador,Masculino,FUNDO ESPECIAL,Recursos de partido político,02/09/2026,3500000.0


# Exercício 9 — Quando o dinheiro entrou

In [42]:
df_receitas["data"] = pd.to_datetime(df_receitas["DT_RECEITA"], format="%d/%m/%Y", errors="coerce")
df_receitas["mes"] = df_receitas["data"].dt.month
df_receitas["mes"].value_counts()

mes
9.0     6987
8.0     6106
10.0     866
1.0        1
4.0        1
12.0       1
Name: count, dtype: int64

Parece estranho ter entrado dinheiro de fundão eleitoral no mês de janeiro, abril e dezembro em 3 casos.

# Exercício 10 — O vazio disfarçado

In [51]:
df_receitas["DS_FONTE_RECEITA"].isna().sum()

mascara_nulo = df_receitas["DS_FONTE_RECEITA"] == "#NULO"
mascara_nulo.sum()

np.int64(79)

# Exercício 11 — Fundão e gênero

In [62]:
mulheres = df_receitas["DS_GENERO"] == "Feminino"
homens = df_receitas["DS_GENERO"] == "Masculino"

df_mulheres = df_receitas[mulheres]
df_homens = df_receitas[homens]

vr_mulheres = df_mulheres["VR_RECEITA"].sum()
vr_homens = df_homens["VR_RECEITA"].sum()
vr_total = df_receitas["VR_RECEITA"].sum()

percentual_mulheres = (vr_mulheres / vr_total) *100
percentual_homens = (vr_homens / vr_total) *100

print(f"O valor total de receitas é R$ {vr_total:.2f}")
print(f"O valor destinado às mulheres é de R$ {vr_mulheres:,.2f}, que representa {percentual_mulheres:.1f}%.")
print(f"O valor destinado aos homens é de R$ {vr_homens:,.2f}, que representa {percentual_homens:.1f}%.")

O valor total de receitas é R$ 320740507.40
O valor destinado às mulheres é de R$ 102,438,685.77, que representa 31.9%.
O valor destinado aos homens é de R$ 218,301,821.63, que representa 68.1%.
